# 2026 FIELD CAMP 주제 1 — Phase 2 관중 입장 수익 최대화 스케줄링
## MILP 코드 제출 양식 (.ipynb)

**팀명:** 6B　　　　　 **팀원:** 변준형, 김○은, 김○준, 박○성, 박○준, 이○빈, 이○민, 최○현　　**제출일:** 2026.07.29

---
### 작성 규칙 (채점에 직접 사용되므로 반드시 지켜주세요)

1. **셀 구조를 유지**하세요. 각 제약식은 ① 마크다운 셀(번호 + LaTeX 수식 + 설명) ② 바로 아래 코드 셀(해당 제약 코드만) 한 쌍으로 작성합니다.
2. 제약식 번호는 함께 제출하는 **모델링 양식(Word)의 번호와 일치**해야 합니다.
3. 하나의 번호에 여러 식이 있으면 수식과 코드 주석에 (1-1), (1-2)처럼 구분합니다.
4. 제약 블록이 부족하면 셀 쌍을 복사해 추가하고, 남으면 삭제하세요.
5. `[예시]`로 표시된 셀은 작성 요령을 보여주기 위한 것으로 Phase 2 문제와 무관합니다. **제출 시 삭제**하세요.
6. 제출 전 **Kernel → Restart & Run All**로 위에서부터 끝까지 오류 없이 실행되는지 확인하세요.
7. 파일명: `[팀명]_phase2_code.ipynb`


## 0. 환경 설정 및 데이터 불러오기

첫 번째 코드 셀에서 필요한 패키지를 설치한 뒤, 수익·구장·거리 데이터를 읽어 전처리합니다.


In [ ]:
##솔버 설치 명령 (주석 해제 후 실행하시면 됩니다.)
%pip install -q pandas docplex cplex

In [ ]:
import pandas as pd
from docplex.mp.model import Model
from docplex.mp.conflict_refiner import ConflictRefiner

print("데이터를 준비 중입니다...")

# 파일 데이터 로드
df_series_rev = pd.read_csv("p_phase2_series.csv")
df_venue = pd.read_csv("team_venue_phase2.csv")
df_dist = pd.read_csv("D_phase2.csv")

# 거리 데이터 전처리
df_dist.columns = df_dist.columns.str.strip()
df_dist["venue"] = df_dist["venue"].str.strip()
df_dist.set_index("venue", inplace=True)

데이터를 준비 중입니다...


## 1. Sets & Indices

| 기호/코드 | 정의 |
|:--|:--|
| $T$, `teams` | 팀 집합 $T=\{1,\dots,10\}$, 팀 인덱스 $i,j,k\in T$ |
| $S$, `series` | 시리즈 집합 $S=\{1,\dots,45\}$, 시리즈 인덱스 $t,\tau\in S$ |
| $W$, `weekends` | 주말 시리즈 집합 $W=\{t\in S\mid t\text{가 홀수}\}$ |
| $\mathcal{V}$, `venues` | 구장 ID 집합 |
| $A$ | 방향이 있는 팀 쌍 집합 $A=\{(i,j)\in T\times T\mid i\ne j\}$ |
| $P$ | 순서가 없는 팀 쌍 집합 $P=\{(i,j)\in T\times T\mid i<j\}$ |
| $S^{\mathrm{move}}$ | 이동 계산 시리즈 집합 $\{2,\dots,45\}$ |
| $S^{\mathrm{gap}}$ | 4개 시리즈 구간의 시작점 집합 $\{1,\dots,42\}$ |
| $S^{(1)}$ | 첫 번째 상대별 홈 편성 구간 $\{1,\dots,18\}$ |
| $S^{(2)}$ | 두 번째 상대별 홈 편성 구간 $\{19,\dots,36\}$ |

> $A$, $P$, $S^{\mathrm{move}}$, $S^{\mathrm{gap}}$, $S^{(1)}$, $S^{(2)}$는 수식의 적용 범위를 명확히 나타내기 위한 표기입니다. 코드에서는 `if i != j`, `if i < j`, `range(...)`로 동일한 범위를 구현합니다.


In [ ]:
# 수식에서는 T=teams, S=series, W=weekends로 표기한다.
# A={(i,j): i!=j}, P={(i,j): i<j}는 코드의 조건문으로 구현한다.
# 시리즈 부분집합은 range(2,46), range(1,43), range(1,19), range(19,37)로 구현한다.

# 팀 및 시리즈 기본 설정
teams = list(range(1, 11))
series = list(range(1, 46))

# 평일/주말 구분 (홀수 시리즈는 주말, 짝수는 평일)
weekends = [t for t in series if t % 2 != 0]

# 구장 집합
venues = list(set(df_venue["venue_id"]))


## 2. Parameters

| 기호/코드 | 정의 · 단위 |
|:--|:--|
| $G_t$, `G[t]` | 시리즈 $t$의 경기 수 [경기/시리즈] |
| $H_{\mathrm{1st}}$, `H_1st` | 주어진 개막 홈 개최 대상 팀 집합 |
| $R_{ijt}$, `R_series[i,j,t]` | 홈팀 $i$, 원정팀 $j$, 시리즈 $t$의 예상 관중 입장 수익 [제공 데이터의 화폐 단위] |
| $V_i$, `V[i]` | 팀 $i$의 홈구장 ID [구장 ID] |
| $D_{vw}$, `D[v,w]` | 구장 $v$에서 구장 $w$까지의 이동거리 [제공 거리 데이터의 단위] |
| $\epsilon$, `epsilon` | 이동거리 편차 페널티 가중치 $0.0001$ |
| $N^{\mathrm{pair}}$ | 동일 두 팀의 총 시리즈 수 기준값 $5$ [시리즈] |
| $N^{\mathrm{game}}$ | 동일 두 팀의 시즌 총 경기 수 기준값 $15$ [경기] |
| $L^{\mathrm{gap}}$ | 재대결 사이에 확보해야 하는 최소 다른 시리즈 수 $3$ [시리즈] |
| $H^{\mathrm{all}}_{\min},H^{\mathrm{all}}_{\max}$ | 시즌 전체 홈시리즈 수 하한·상한 $22,23$ [시리즈] |
| $H^{W}_{\min},H^{W}_{\max}$ | 주말 홈시리즈 수 하한·상한 $11,12$ [시리즈] |
| $\Delta^{\mathrm{dist}}$ | 팀 간 총 이동거리 편차 상한 $4500$ [거리 데이터와 동일 단위] |
| $t^{\mathrm{open}},t^{\mathrm{post}}$ | 개막 및 올스타 직후 시리즈 번호 $1,31$ |

> 코드에서는 일부 기준값을 별도 상수 변수로 선언하지 않고 제약식 안에 숫자로 직접 사용합니다. 위 표는 실제 코드에 사용된 입력값과 기준값을 정리한 것입니다.


In [ ]:
#[제약식 2] G_t는 기본 3경기, 개막 2경기, 31번 시리즈 4경기로 정의한다.
# 또한 제약식 4(31번 시리즈=개막 시리즈),
# 제약식 5(동일 상대 총 5시리즈)와 G_t 정의로도 자동 성립한다.
# 아래 입력값의 단위는 데이터 정의를 따른다.
# G: 경기/시리즈, R_series: 예상 수익, D: 이동거리, V: 구장 ID
# 제약식에 직접 사용되는 기준값: 5시리즈, 최소 간격 3시리즈,
# 전체 홈 22~23, 주말 홈 11~12, 이동거리 편차 상한 4500

# 1️⃣ 시리즈의 경기 수는 기본 3, 개막(1번 시리즈) 2, 올스타 직후(31번 시리즈) 4로 정한다.
G = {t: 3 for t in series}
G[1] = 2
G[31] = 4

# 2️⃣ 개막 홈 개최는 직전 시즌 상위 5위팀 (단, Rams(2)와 Jaguars(1)가 모두 5위 안이면 순위가 높은 1팀이 개최, 6위 팀 대체)
# 1위~6위 중 2위 제외: Jaguars(1), Titans(3), Dolphins(4), Ravens(5), Chiefs(6)
H_1st = [1, 3, 4, 5, 6]

# 시리즈별 예상 수익
R_series = {}
for _, row in df_series_rev.iterrows():
    R_series[(row["home_team_id"], row["away_team_id"], row["series"])] = row["p"]

# 팀 ID -> 구장 ID 매핑
V = {row["team_id"]: row["venue_id"] for _, row in df_venue.iterrows()}
venue_name_to_id = {
    row["venue_name"].strip(): row["venue_id"]
    for _, row in df_venue.iterrows()
}

# 구장 간 이동 거리 D 세팅
D = {}
for origin in df_dist.index:
    for dest in df_dist.columns:
        origin_id = venue_name_to_id[origin]
        dest_id = venue_name_to_id[dest]
        D[(origin_id, dest_id)] = df_dist.loc[origin, dest]

for v in venues:
    if (v, v) not in D:
        D[(v, v)] = 0


## 3. Decision Variables

### 경기 편성 변수

$$
x_{ijt}\in\{0,1\}
\qquad \forall (i,j)\in A,\;t\in S
$$

시리즈 $t$에 팀 $i$가 홈, 팀 $j$가 원정이면 1입니다.

### 홈 여부 파생식

$$
h_{it}=\sum_{j\in T:j\ne i}x_{ijt}
\qquad \forall i\in T,\;t\in S
$$

시리즈 $t$에 팀 $i$가 홈이면 1이 되는 **파생 선형식**이며, 별도의 결정변수는 아닙니다.

### 시리즈별 위치 파생식

$$
y_{ikt}=
\begin{cases}
h_{it}, & i=k,\\
x_{kit}, & i\ne k
\end{cases}
\qquad \forall i,k\in T,\;t\in S
$$

팀 $i$가 시리즈 $t$에 팀 $k$의 홈구장에 위치하는지를 나타내는 파생 선형식입니다. 코드에서는 `get_host_location(i, k, t)`로 구현합니다.

### 시리즈 간 이동 흐름 변수

$$
z_{ijkt}\in[0,1]
\qquad \forall i,j,k\in T,\;t\in S^{\mathrm{move}}
$$

팀 $i$가 시리즈 $t-1$의 팀 $j$ 구장에서 시리즈 $t$의 팀 $k$ 구장으로 이동하는 네트워크 흐름 변수입니다.

### 총 이동거리 및 편차 변수

$$
\mathrm{dist}_i\in\mathbb{R}_{+}
\qquad \forall i\in T
$$

$$
d_{\max},d_{\min}\in\mathbb{R}_{+}
$$

$\mathrm{dist}_i$는 팀 $i$의 시즌 총 이동거리이며, $d_{\max}$와 $d_{\min}$은 팀별 총 이동거리의 최댓값과 최솟값을 연결합니다.


In [ ]:
# [조건 1️⃣] x는 시리즈 단위 변수이므로 한 시리즈 안에서 매치업·홈/원정·구장이 고정된다.
# [조건 6️⃣ / 제약식 12] z와 get_host_location은 시리즈 사이의 이동만 추적한다.
# x의 적용 범위: (i,j) in A, t in S
# h는 별도 결정변수가 아니라 x로 정의되는 파생 선형식
# z의 적용 범위: i,j,k in T, t in S_move={2,...,45}
# get_host_location은 수식의 y_{ikt}에 해당함

print("CPLEX 모델을 구축 중입니다...")
mdl = Model(name="Baseball_League_Scheduling")

# 1️⃣ 한 시리즈 안에서는 매치업(홈 팀, 원정 팀)과 구장을 고정한다.
# (시리즈 단위로 홈/원정 i, j를 결정하는 변수 x에 의해 자동 만족됨)
x = mdl.binary_var_dict(
    [(i, j, t) for i in teams for j in teams if i != j for t in series],
    name="x"
)

# [최적화 적용] h 변수를 결정 변수가 아닌 수식(Expression)으로 정의하여 변수와 제약식 절약
h = {
    (i, t): mdl.sum(x[i, j, t] for j in teams if i != j)
    for i in teams for t in series
}

# 6️⃣ 이동을 추적하기 위한 Network Flow 전이 변수 (연속 변수)
z = mdl.continuous_var_dict(
    [
        (i, j, k, t)
        for i in teams
        for j in teams
        for k in teams
        for t in range(2, 46)
    ],
    lb=0,
    ub=1,
    name="z"
)

dist = mdl.continuous_var_dict(teams, name="dist")
d_max = mdl.continuous_var(name="d_max")
d_min = mdl.continuous_var(name="d_min")

# 헬퍼 함수: 팀 i가 시리즈 t에 구단 k의 홈 구장에 위치하는지 여부 반환
def get_host_location(i, k, t):
    if i == k:
        return h[i, t]       # 홈 경기인 경우
    else:
        return x[k, i, t]    # k팀의 홈구장에서 원정 경기를 하는 경우


CPLEX 모델을 구축 중입니다...


## 4. Objective Function — 관련 조건 6️⃣, 제약식 13과 연계

$$
\max Z=
\sum_{(i,j)\in A}\sum_{t\in S}R_{ijt}x_{ijt}
-\epsilon\left(d_{\max}-d_{\min}\right)
$$

**적용 범위:** $(i,j)\in A,\;t\in S$

**설명:** 예상 관중 입장 수익을 주된 항으로 최대화하면서, $\epsilon=0.0001$의 작은 가중치로 팀 간 이동거리 편차가 작은 해를 선호합니다. 이 구현은 단일 가중합 목적함수이므로 수익을 사전식으로 완전히 고정한 뒤 편차를 최소화하는 방식과는 다릅니다.


In [ ]:
# [조건 6️⃣ / 제약식 12 연계 목적함수 보조항] 팀 간 최대 이동거리 편차가 작을수록 선호하도록 작은 epsilon 페널티를 적용한다.
# epsilon은 거리 편차에 작은 페널티를 주지만, 사전식 다중목적 최적화는 아니다.

# 6️⃣ 팀 간 최대 편차는 목적함수를 작을수록 좋지만,
# 예상 수익을 주된 항으로 최대화하면서, 작은 eplison 가중치로 이동거리 편차가 작은 해를 선호한다. (단일 가중합 목적함수)
epsilon = 0.0001

total_revenue = mdl.sum(
    R_series[i, j, t] * x[i, j, t]
    for i in teams
    for j in teams if i != j
    for t in series
)

distance_penalty = epsilon * (d_max - d_min)

# 목적함수: 관중 입장 수익 최대화 (거리 편차 페널티 차감)
mdl.maximize(total_revenue - distance_penalty)


## 5. Constraints

아래 제약식은 문제 조건 1️⃣~6️⃣과의 관련성을 함께 표시하고, 전체 제약을 **제약식 1부터 제약식 13까지 연속 번호**로 구분합니다. 하나의 제약식에 여러 관계식이 포함되는 경우에도 동일한 제약식 번호 안에 함께 표시합니다.


### 제약식 1 — 매 시리즈 모든 팀 참가 (관련 조건: 1️⃣)

$$
\sum_{j\in T:j\ne i}\left(x_{ijt}+x_{jit}\right)=1
\qquad \forall i\in T,\;t\in S
\tag{1}
$$

**적용 범위:** 모든 $i\in T,\;t\in S$

**설명:** 각 팀은 매 시리즈에서 홈 또는 원정으로 정확히 한 번 참가합니다. 따라서 각 시리즈에는 10개 구단이 모두 참가하고 총 5개의 매치업이 편성됩니다.


In [7]:
# [조건 1️⃣ / 제약식 1] 매 시리즈에서 10개 구단이 모두 정확히 한 번 참가한다.
# [추가 표기] 적용 범위: 모든 i in teams, t in series
# 제약식 1: 매 시리즈 모든 팀 1회 참가
for t in series:
    for i in teams:
        mdl.add_constraint(
            mdl.sum(x[i, j, t] + x[j, i, t] for j in teams if i != j) == 1,
            f"PlayOnce_{i}_{t}"
        )


### 제약식 2 — 시리즈별 경기 수 및 상대별 총 15경기 성립 (관련 조건: 1️⃣)

$$
\begin{aligned}
G_t&=
\begin{cases}
2, & t=1,\\
4, & t=31,\\
3, & \text{그 외},
\end{cases}\\[4pt]
\sum_{t\in S}G_t\left(x_{ijt}+x_{jit}\right)&=15
\qquad \forall (i,j)\in P
\end{aligned}
\tag{2}
$$

**적용 범위:** 모든 $t\in S$ 및 모든 $(i,j)\in P$

**설명:** 동일 상대와 총 15경기를 치르도록 코드의 제약식 (2)에 직접 구현한다. 또한 제약식 (4)에 따라 31번 시리즈가 개막 시리즈와 동일한 매치업 및 제약식 (5)의 총 5시리즈 조건 및 G(t) 정의에 의해서도 자동으로 성립함.

**구현 방식:** 시리즈 예상 수익 데이터는 시리즈 단위이므로 $G_t$를 목적함수에 다시 곱하지 않습니다. 제약의 명시적 표현을 위해서 포함했지만, **제약식 5, 제약식 6 및 $G_t$ 정의의 결합**으로 성립시킵니다.


In [ ]:
# [조건 1️⃣ / 제약식 2]
# G_t는 Parameters 코드 셀에서 정의되어 있다.
# 상대별 총 15경기는 제약식 5(31번=개막)와 제약식 6(총 5시리즈)의 결합으로 자동 충족하지만, 모델링 문서와의 명시적 일치를 위해 추가한다.

for i in teams:
    for j in teams:
        if i < j:
            mdl.add_constraint(
                mdl.sum(
                    G[t] * (x[i, j, t] + x[j, i, t])
                    for t in series
                ) == 15,
                f"Total15Games_{i}_{j}"
            )

### 제약식 3 — 개막 홈 개최 팀 지정 (관련 조건: 2️⃣)

$$
\begin{aligned}
h_{i1}&=1 && \forall i\in H_{\mathrm{1st}},\\
h_{i1}&=0 && \forall i\in T\setminus H_{\mathrm{1st}}
\end{aligned}
\tag{3}
$$

**적용 범위:** 모든 $i\in T$

**설명:** `team_venue_phase2.csv`의 `team_id` 순위를 기준으로 정한 개막 홈 개최 대상 팀만 1번 시리즈를 홈에서 치릅니다. Metro Field를 공유하는 Jaguars와 Rams가 모두 상위 5위 안에 포함되는 경우에는 순위가 높은 팀만 개최하고 6위 팀이 남은 자리를 대체합니다.


In [ ]:
# [조건 2️⃣ / 제약식 3] 개막 홈 개최 팀을 H_1st로 지정한다. (적용 범위: 모든 i in teams)
# 제약식 3: 개막 홈 개최 팀 지정
for i in teams:
    if i in H_1st:
        mdl.add_constraint(h[i, 1] == 1, f"OpenHome_{i}")
    else:
        mdl.add_constraint(h[i, 1] == 0, f"OpenAway_{i}")


### 제약식 4 — 31번 시리즈는 개막과 동일한 매치업·홈/원정 (관련 조건: 2️⃣)

$$
x_{ij,31}=x_{ij,1}
\qquad \forall (i,j)\in A
\tag{4}
$$

**적용 범위:** 모든 $(i,j)\in A$

**설명:** 31번 시리즈는 개막 시리즈와 동일한 대진과 동일한 홈·원정 배치를 사용합니다.


In [ ]:
# [조건 2️⃣ / 제약식 4] 31번 시리즈를 개막과 동일한 매치업·홈/원정으로 고정한다. (적용 범위: 모든 i,j in teams, i!=j)
# 제약식 4: 31번 시리즈는 개막전과 동일 대진
for i in teams:
    for j in teams:
        if i != j:
            mdl.add_constraint(
                x[i, j, 31] == x[i, j, 1],
                f"SameAsOpening_{i}_{j}"
            )


### 제약식 5 — 동일 두 팀 총 5시리즈 (관련 조건: 3️⃣)

$$
\sum_{t\in S}\left(x_{ijt}+x_{jit}\right)=5
\qquad \forall (i,j)\in P
\tag{5}
$$

**적용 범위:** 모든 $(i,j)\in P$

**설명:** 같은 두 팀은 시즌 중 총 5번의 시리즈를 치릅니다.


In [ ]:
# [조건 3️⃣ / 제약식 5] 같은 두 팀은 시즌 중 총 5시리즈를 치른다. (적용 범위: 모든 팀 쌍 i<j)
# 제약식 5: 동일 두 팀 총 5시리즈
for i in teams:
    for j in teams:
        if i < j:
            # 제약식 2의 15경기 조건은 본 제약과 제약식 4, 그리고 G_t 정의로도 자동 성립한다.
            mdl.add_constraint(
                mdl.sum(x[i, j, t] + x[j, i, t] for t in series) == 5,
                f"Total5Series_{i}_{j}"
            )


### 제약식 6 — 재대결 최소 간격 (관련 조건: 3️⃣)

$$
\sum_{\tau=t}^{t+3}\left(x_{ij\tau}+x_{ji\tau}\right)\le1
\qquad \forall (i,j)\in P,\;t\in S^{\mathrm{gap}}
\tag{6}
$$

**적용 범위:** 모든 $(i,j)\in P,\;t\in S^{\mathrm{gap}}$

**설명:** 같은 두 팀은 연속으로 시리즈를 치를 수 없으며, 재대결 전에는 최소 3개의 다른 시리즈가 배치되어야 합니다.


In [ ]:
# [조건 3️⃣ / 제약식 6] 같은 두 팀의 재대결 전 최소 3개 시리즈 간격을 확보한다. 
# (적용 범위: 모든 팀 쌍 i<j, 시작 시리즈 t=1,...,42 / 마크다운 수식에서는 시리즈 합산 인덱스를 tau로 표시함)
# 제약식 6: 재대결 최소 간격
for i in teams:
    for j in teams:
        if i < j:
            for t in range(1, 43):
                end_t = min(t + 3, 45)
                mdl.add_constraint(
                    mdl.sum(
                        x[i, j, k] + x[j, i, k]
                        for k in range(t, end_t + 1)
                    ) <= 1,
                    f"Gap3_{i}_{j}_{t}"
                )


### 제약식 7 — 구간별 상대 홈·원정 시리즈 각 1회 (관련 조건: 4️⃣)

$$
\begin{aligned}
\sum_{t\in S^{(1)}}x_{ijt}&=1 && \forall (i,j)\in A,\\
\sum_{t\in S^{(2)}}x_{ijt}&=1 && \forall (i,j)\in A
\end{aligned}
\tag{7}
$$

**적용 범위:** 모든 $(i,j)\in A$

**설명:** 시리즈 1~18과 19~36의 각 구간에서 각 팀은 모든 상대를 홈으로 한 번씩 상대합니다. 반대 방향의 변수에도 같은 식이 적용되므로 상대별 원정 시리즈도 각 구간에서 한 번씩 편성됩니다. 다섯 번째 시리즈의 홈·원정은 상대별로 미리 지정하지 않습니다.


In [ ]:
# [조건 4️⃣ / 제약식 7] 1~18 및 19~36 구간에서 상대별 홈·원정 시리즈를 각각 1회 편성한다. 첫 번째 식은 시리즈 1~18, 두 번째 식은 시리즈 19~36에 적용한다.
# 적용 범위: 모든 i,j in teams, i!=j
# 제약식 7: 구간별 상대 홈시리즈 1회
for i in teams:
    for j in teams:
        if i != j:
            mdl.add_constraint(
                mdl.sum(x[i, j, t] for t in range(1, 19)) == 1,
                f"Phase1_{i}_{j}"
            )
            mdl.add_constraint(
                mdl.sum(x[i, j, t] for t in range(19, 37)) == 1,
                f"Phase2_{i}_{j}"
            )


### 제약식 8 — 시즌 전체 및 주말 홈시리즈 수 (관련 조건: 4️⃣)

$$
\begin{aligned}
22&\le\sum_{t\in S}h_{it}\le23 && \forall i\in T,\\
11&\le\sum_{t\in W}h_{it}\le12 && \forall i\in T
\end{aligned}
\tag{8}
$$

**적용 범위:** 모든 $i\in T$

**설명:** 각 팀은 시즌 전체 홈 시리즈를 22~23개, 주말 홈 시리즈를 11~12개 진행합니다.


In [ ]:
# [조건 4️⃣ / 제약식 8] 전체 홈 22~23개 및 주말 홈 11~12개를 적용한다.
# [추가 표기] 첫 번째 범위는 전체 홈, 두 번째 범위는 주말 홈에 적용한다.
# [추가 표기] 적용 범위: 모든 i in teams
# 제약식 8: 시즌 및 주말 홈시리즈 수
for i in teams:
    mdl.add_constraint(
        mdl.sum(h[i, t] for t in series) >= 22,
        f"TotalHomeMin_{i}"
    )
    mdl.add_constraint(
        mdl.sum(h[i, t] for t in series) <= 23,
        f"TotalHomeMax_{i}"
    )
    mdl.add_constraint(
        mdl.sum(h[i, t] for t in weekends) >= 11,
        f"WeekendHomeMin_{i}"
    )
    mdl.add_constraint(
        mdl.sum(h[i, t] for t in weekends) <= 12,
        f"WeekendHomeMax_{i}"
    )


### 제약식 9 — 홈·원정 4연속 금지 (관련 조건: 5️⃣)

$$
\begin{aligned}
\sum_{\tau=t}^{t+3}h_{i\tau}&\le3 && \forall i\in T,\;t\in S^{\mathrm{gap}},\\
\sum_{\tau=t}^{t+3}\sum_{j\in T:j\ne i}x_{ji\tau}&\le3 && \forall i\in T,\;t\in S^{\mathrm{gap}}
\end{aligned}
\tag{9}
$$

**적용 범위:** 모든 $i\in T,\;t\in S^{\mathrm{gap}}$

**설명:** 어떤 팀도 홈 시리즈 또는 원정 시리즈를 4번 연속 가질 수 없으며, 최대 3개 연속까지만 허용됩니다.


In [ ]:
# [조건 5️⃣ / 제약식 9] 홈 또는 원정 시리즈의 4연속 편성을 금지한다.
# [추가 표기] 첫 번째 식은 홈 4연속, 두 번째 식은 원정 4연속 금지이다.
# [추가 표기] 적용 범위: 모든 i in teams, 시작 시리즈 t=1,...,42
# 제약식 9: 홈·원정 4연속 금지
for i in teams:
    for t in range(1, 43):
        mdl.add_constraint(
            mdl.sum(h[i, k] for k in range(t, t + 4)) <= 3,
            f"No4Home_{i}_{t}"
        )
        # [최적화 적용] x 변수를 직접 활용해 식 단순화
        mdl.add_constraint(
            mdl.sum(
                x[j, i, k]
                for j in teams if i != j
                for k in range(t, t + 4)
            ) <= 3,
            f"No4Away_{i}_{t}"
        )


### 제약식 10 — 공유 구장 동시 홈경기 금지 (관련 조건: 5️⃣)

$$
h_{1t}+h_{2t}\le1
\qquad \forall t\in S
\tag{10}
$$

**적용 범위:** 모든 $t\in S$

**설명:** Metro Field를 공유하는 Jaguars(1)와 Rams(2)가 같은 시리즈에 동시에 홈이 되는 일정을 금지합니다. 따라서 같은 구장에서 같은 시리즈에 두 경기가 열리지 않습니다.


In [ ]:
# [조건 5️⃣ / 제약식 10] Metro Field 공유 팀의 같은 시리즈 동시 홈을 금지한다. (적용 범위: 모든 t in series)
# 제약식 10: 공유 구장 동시 홈경기 금지
for t in series:
    mdl.add_constraint(
        h[1, t] + h[2, t] <= 1,
        f"StadiumShare_{t}"
    )


### 제약식 11 — 시리즈 사이의 이동 흐름 계산 (관련 조건: 6️⃣)

팀 $i$가 시리즈 $t$에 팀 $k$의 홈구장에 위치하는지를 $y_{ikt}$로 나타냅니다.

$$
\begin{aligned}
\sum_{k\in T}z_{ijkt}&=y_{ij,t-1} && \forall i,j\in T,\;t\in S^{\mathrm{move}},\\
\sum_{j\in T}z_{ijkt}&=y_{ikt} && \forall i,k\in T,\;t\in S^{\mathrm{move}}
\end{aligned}
\tag{11}
$$

**적용 범위:** 모든 $i,j,k\in T,\;t\in S^{\mathrm{move}}$

**설명:** 팀의 이동은 시리즈와 시리즈 사이에서만 발생한다고 계산합니다. 시리즈 도중에는 구장이 바뀌지 않아 이동이 없으며, 개막 시리즈 이전의 이동은 포함하지 않으므로 $t=2$부터 계산합니다.


### 제약식 12 — 팀별 총 이동거리 및 최대 편차 제한 (관련 조건: 6️⃣)

$$
\begin{aligned}
\mathrm{dist}_i
&=\sum_{t\in S^{\mathrm{move}}}\sum_{j\in T}\sum_{k\in T}
D_{V_jV_k}z_{ijkt}
&& \forall i\in T,\\
d_{\max}&\ge\mathrm{dist}_i,
\quad d_{\min}\le\mathrm{dist}_i
&& \forall i\in T,\\
d_{\max}-d_{\min}&\le4500
\end{aligned}
\tag{12}
$$

**적용 범위:** 팀별 거리식과 최대·최소 연결식은 모든 $i\in T$에 적용하며, 편차 상한은 단일 제약입니다.

**설명:** 팀의 시즌 총 이동거리는 시리즈 사이 이동거리의 합입니다. 같은 구장이 연속되면 $D_{vv}=0$이므로 이동거리는 0이며, 팀별 총 이동거리의 최댓값과 최솟값 차이는 4500 이하여야 합니다.


In [ ]:
# [조건 6️⃣ / 제약식 11] 이동은 시리즈 사이에서만 계산하고 개막 이전 이동은 제외한다.
# [조건 6️⃣ / 제약식 12] 팀별 총 이동거리와 최대·최소 편차 4500 상한을 적용한다.
# 제약식 12: 팀 간 이동거리 편차 상한
mdl.add_constraint(d_max - d_min <= 4500, "DistDiffLimit")

# 제약식 11: 시리즈 간 위치 전이 흐름 연결
# 팀의 이동은 시리즈와 시리즈 사이에서만 발생하는 것으로 계산한다.
# 개막 시리즈 이전의 이동은 계산에 포함하지 않으므로 t=2부터 시작
for i in teams:
    for t in range(2, 46):
        # 제약식 11: 이전 위치별 유출 흐름
        # Outflow: t-1시리즈에 j구장에 있던 팀 i가 t시리즈에서 다른 구장으로 이동
        for j in teams:
            mdl.add_constraint(
                mdl.sum(z[i, j, k, t] for k in teams)
                == get_host_location(i, j, t - 1),
                f"OutFlow_{i}_{j}_{t}"
            )

        # 제약식 11: 현재 위치별 유입 흐름
        # Inflow: t시리즈에 k구장에 도착한 팀 i는
        # t-1시리즈에서 어딘가로부터 옴
        for k in teams:
            mdl.add_constraint(
                mdl.sum(z[i, j, k, t] for j in teams)
                == get_host_location(i, k, t),
                f"InFlow_{i}_{k}_{t}"
            )

# 제약식 12: 팀별 총 이동거리 및 최대·최소 이동거리 연결
# 팀의 시즌 총 이동거리는 이동들의 거리 합
for i in teams:
    mdl.add_constraint(
        dist[i] == mdl.sum(
            D[V[j], V[k]] * z[i, j, k, t]
            for j in teams
            for k in teams if V[j] != V[k]
            for t in range(2, 46)
        ),
        f"DistCalc_{i}"
    )
    mdl.add_constraint(d_max >= dist[i], f"MaxDist_{i}")
    mdl.add_constraint(d_min <= dist[i], f"MinDist_{i}")


### 제약식 13 — 변수 범위

$$
\begin{aligned}
x_{ijt}&\in\{0,1\} && \forall (i,j)\in A,\;t\in S,\\
0\le z_{ijkt}&\le1 && \forall i,j,k\in T,\;t\in S^{\mathrm{move}},\\
\mathrm{dist}_i&\ge0 && \forall i\in T,\\
d_{\max}&\ge0,\quad d_{\min}\ge0
\end{aligned}
\tag{13}
$$

**설명:** 변수 범위는 별도의 `add_constraint` 문이 아니라 결정변수 선언 단계의 변수 종류와 상·하한으로 처리합니다. $h_{it}$와 $y_{ikt}$는 별도의 결정변수가 아닌 파생 선형식입니다.


In [ ]:
# [제약식 13] 문제 조건 1️⃣~6️⃣에 사용되는 변수의 정의역을 선언부에서 처리한다.
# 제약식 13: 변수 범위
# x: binary_var_dict 선언으로 이진 범위 처리
# z: continuous_var_dict의 lb=0, ub=1로 범위 처리
# dist, d_max, d_min: docplex 연속변수의 기본 하한 0으로 비음수 범위 처리
# h, get_host_location(y): x로부터 계산되는 파생 선형식이므로 별도 변수 범위 제약 없음


## 6. 모델 실행

솔버 파라미터를 설정하고 최적화를 실행한 뒤, 스케줄 CSV를 저장합니다.


In [ ]:
print("최고 수익 달성을 위한 최적화를 시작합니다...")

mdl.parameters.timelimit = 28800           # 8시간 제한
mdl.parameters.threads = 0                 # 모든 CPU 코어 사용

# 1. MIP Gap 하향
mdl.parameters.mip.tolerances.mipgap = 0.0001

# 2. 탐색 전략: 실행가능해의 품질개선에 집중
mdl.parameters.emphasis.mip = 1

# 3. CPLX 자동 조정 사용
mdl.parameters.mip.strategy.rinsheur = 0
mdl.parameters.mip.strategy.heuristicfreq = 0

# 4. 솔루션 폴리싱 (Solution Polishing) 가동
mdl.parameters.mip.polishafter.time = 21600

log_filename = "phase2_log.txt"
with open(log_filename, "w", encoding="utf-8") as f:
    solution = mdl.solve(log_output=f)

if solution:
    print("\n[최적해 도출 성공!]")
    # get_objective_value()는 순수 예상 수익이 아니라
    # total_revenue - epsilon * (d_max - d_min)의 값을 반환한다.
    print(f"예상 총수익(Objective Value): {solution.get_objective_value():.2f}")
    print(
        f"최대 이동거리: {solution.get_value(d_max):.2f}, "
        f"최소 이동거리: {solution.get_value(d_min):.2f}"
    )

    schedule_data = []
    for t in series:
        for i in teams:
            for j in teams:
                if i != j:
                    if solution.get_value(x[i, j, t]) > 0.5:
                        schedule_data.append({
                            "series": t,
                            "home_team_id": i,
                            "away_team_id": j
                        })

    df_schedule = pd.DataFrame(schedule_data)
    csv_filename = "6B_phase2_schedule.csv"
    df_schedule.to_csv(
    "6B_phase2_schedule.csv",
    index=False,
    encoding="utf-8-sig"
    )
    print(f"\n스케줄이 '{csv_filename}' 파일로 성공적으로 생성되었습니다.")

else:
    print(
        "\n[Infeasible] 해를 찾지 못했습니다. "
        "Conflict Refiner를 실행하여 충돌 원인을 분석합니다..."
    )
    cr = ConflictRefiner()
    conflicts = cr.refine_conflict(mdl)
    print("\n=== 충돌(Conflict)이 발생한 제약 조건 ===")
    cr.display_conflicts(conflicts)


최고 수익 달성을 위한 최적화를 시작합니다...

[최적해 도출 성공!]
예상 총수익(Objective Value): 78292.66
최대 이동거리: 9420.00, 최소 이동거리: 6045.00

스케줄이 '6B_phase2_schedule.csv' 파일로 성공적으로 생성되었습니다.


In [4]:
## 위의 예상 총수익에는 미소 가중치로 패널티가 반영되어있으므로 패널티를 복원한 순수 예상 총수익을 출력한다.

import pandas as pd

schedule = pd.read_csv(
    "6B_phase2_schedule.csv",
    encoding="utf-8-sig"
)

revenue = pd.read_csv("p_phase2_series.csv")

result = schedule.merge(
    revenue[
        ["series", "home_team_id", "away_team_id", "p"]
    ],
    on=["series", "home_team_id", "away_team_id"],
    how="left",
    validate="many_to_one"
)

if result["p"].isna().any():
    missing = result[result["p"].isna()]
    raise ValueError(
        f"수익 데이터와 연결되지 않은 일정이 있습니다:\n{missing}"
    )

pure_total_revenue = result["p"].sum()

print(f"순수 예상 총수익: {pure_total_revenue:.2f}")


순수 예상 총수익: 78293.00


## 7. 결과 — 경기 일정 출력 (제출 파일)

위 모델 실행 셀에서 해가 존재하면 `series`, `home_team_id`, `away_team_id` 세 컬럼으로 경기 일정을 추출합니다.

| 항목 | baseline 제출 규격 |
|:--|:--|
| 파일명 | `6B_phase2_schedule.csv` |
| 컬럼 순서 | `series`, `home_team_id`, `away_team_id` |
| 행 수 | 정확히 225행 |
| 값 형식 | 세 컬럼 모두 정수 |
| 권장 정렬 | `series`, `home_team_id` 오름차순 |
| 인코딩 | `utf-8-sig` |



## 8. 자가 검증 체크리스트 (권장)
제출 전, 도출된 일정이 각 조건을 만족하는지 확인하고 표를 채우세요.

| 문제 조건 (상세: 교육자료 3) | 반영 제약식 번호 | 결과 일정에서 확인 (O/X) |
|:---------------------------------------------|:---------------:|:----------------------:|
| 조건 ① 시리즈 편성 기본 구조 | 1, 2 <br>※ 제약식 2는 제약식 4·5와 G(t) 정의로도 자동 성립한다. | O |
| 조건 ② 개막 시리즈 · 올스타 직후 4연전 규칙 | 3, 4 | O |
| 조건 ③ 동일 매치업 대진 횟수 · 최소 간격 | 5, 6 | O |
| 조건 ④ 홈 · 원정 밸런스 (라운드/시즌/주말) | 7, 8 | O |
| 조건 ⑤ 연속 홈 · 원정 상한 및 구장 중복 방지 | 9, 10 | O |
| 조건 ⑥ 팀 간 이동거리 편차 상한 | 11, 12<br>※ 제약식 12은 목적함수의 거리 편차 보조항과 연계 | O |

